# 03 — From Baselines to Statistical Models

This notebook walks through a **progressive model comparison** on time-series
forecasting, starting from the simplest baselines and building up to ARIMA and
ensemble (SCUM) models.  Along the way we evaluate every model with standard
error metrics, cross-validation, and the Kaboudan metric.

**Data**: M4 Hourly subset (series H1, H2, H3)  
**Libraries**: `polars-ts`, `statsforecast`, `hvplot`

| Stage | Models |
|---|---|
| Baselines | Naive, Seasonal Naive, Moving Average |
| Spectral | FFT |
| Exponential Smoothing | SES, Holt, Holt-Winters |
| ARIMA family | ARIMA(1,1,1), Auto-ARIMA |
| Ensemble | SCUM (via StatsForecast) |

**References**  
- Hyndman & Athanasopoulos, [*Forecasting: Principles and Practice*](https://otexts.com/fpp3/)  
- Makridakis et al., *The M4 Competition* (2020)  
- Kaboudan, M. A., *A measure of time series' predictability* (1999)

In [ ]:
# Install polars-timeseries and hvplot if not already available
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]
if importlib.util.find_spec("hvplot") is None:
    %pip install -q hvplot

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization

import polars as pl
from statsforecast import StatsForecast

import polars_ts as pts  # noqa  — registers the .pts namespace
from polars_ts import (
    SCUM,
    arima_fit,
    arima_forecast,
    auto_arima,
    expanding_window_cv,
    fft_forecast,
    holt_forecast,
    holt_winters_forecast,
    mae,
    mape,
    mase,
    moving_average_forecast,
    naive_forecast,
    rmse,
    seasonal_naive_forecast,
    ses_forecast,
    smape,
)

## 1 — Data Loading & Train / Test Split

We load three hourly series from the M4 competition hosted on S3 and
split each series 80 / 20 by time.

In [ ]:
df = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").is_in(["H1", "H2", "H3"]))
    .collect()
)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
HORIZON = 48
SEASON_LENGTH = 24

# 80/20 split per series
split_idx = (
    df.group_by("unique_id")
    .agg(pl.col("ds").count().alias("n"))
    .with_columns((pl.col("n") * 0.8).cast(pl.Int64).alias("split"))
)

train = (
    df.join(split_idx, on="unique_id")
    .with_columns(pl.col("ds").cum_count().over("unique_id").alias("_row"))
    .filter(pl.col("_row") <= pl.col("split"))
    .select("unique_id", "ds", "y")
)

test = (
    df.join(split_idx, on="unique_id")
    .with_columns(pl.col("ds").cum_count().over("unique_id").alias("_row"))
    .filter(pl.col("_row") > pl.col("split"))
    .select("unique_id", "ds", "y")
)

# Trim test to the forecast horizon for fair comparison
test = (
    test.with_columns(pl.col("ds").cum_count().over("unique_id").alias("_row"))
    .filter(pl.col("_row") <= HORIZON)
    .select("unique_id", "ds", "y")
)

print(f"Train: {train.shape},  Test: {test.shape}")

## 2 — Baseline Models

Three dead-simple forecasts that any more sophisticated model should beat:

| Model | Idea |
|---|---|
| **Naive** | Repeat the last observed value |
| **Seasonal Naive** | Repeat the last observed *season* |
| **Moving Average** | Forecast the rolling mean of the last *k* observations |

In [ ]:
# Convert 'ds' column to datetime type
# M4 hourly 'ds' contains integer indices; map them to an arbitrary datetime epoch.
base_datetime = pl.datetime(2000, 1, 1, 0, 0, 0)
train_datetime = (
    train.with_columns((base_datetime + pl.duration(hours=pl.col("ds") - 1)).alias("ds_datetime"))
    .drop("ds")
    .rename({"ds_datetime": "ds"})
)

fc_naive = naive_forecast(train_datetime, h=HORIZON)
fc_snaive = seasonal_naive_forecast(train_datetime, h=HORIZON, season_length=SEASON_LENGTH)
fc_ma = moving_average_forecast(train_datetime, h=HORIZON, window_size=SEASON_LENGTH)

# Quick sanity check
fc_naive.head(3)

## 3 — FFT Forecast

The FFT approach decomposes the signal in the frequency domain, keeps only
the top *n* harmonics, and projects them forward.  It works well when the
series has strong, stable periodicity.

In [ ]:
fc_fft = fft_forecast(train_datetime, h=HORIZON, n_harmonics=5)
fc_fft.head(3)

## 4 — Exponential Smoothing

We step up to adaptive models that place exponentially decaying weights on
past observations.

| Model | Components |
|---|---|
| **SES** | Level only |
| **Holt** | Level + Trend |
| **Holt-Winters** | Level + Trend + Seasonality |

In [ ]:
fc_ses = ses_forecast(train, h=HORIZON, alpha=0.3)
fc_holt = holt_forecast(train, h=HORIZON, alpha=0.3, beta=0.1)
fc_hw = holt_winters_forecast(train, h=HORIZON, season_length=SEASON_LENGTH)

fc_hw.head(3)

## 5 — ARIMA Family

ARIMA models capture autoregressive and moving-average dynamics after
differencing.  We try a manual ARIMA(1,1,1) and then let `auto_arima`
select the best order automatically.

In [ ]:
# Manual ARIMA(1,1,1)
fitted_models = arima_fit(train, order=(1, 1, 1))
fc_arima = arima_forecast(fitted_models, h=HORIZON)

# Auto-ARIMA — searches over (p,d,q) x (P,D,Q,s) space
fc_auto = auto_arima(train, h=HORIZON, season_length=SEASON_LENGTH)

fc_auto.head(3)

## 6 — SCUM Ensemble (via StatsForecast)

**SCUM** stands for *Seasonal-naive, CES, Unobserved-components, MSTL*.
It is a lightweight ensemble provided by `polars-ts` that wraps StatsForecast.

In [ ]:
sf = StatsForecast(
    models=[SCUM(season_length=SEASON_LENGTH)],
    freq=1,
)

fc_scum = sf.forecast(df=train, h=HORIZON)
fc_scum = fc_scum.rename({"SCUM": "y_hat"})

fc_scum.head(3)

## 7 — Forecast Visualization

Let us overlay the actual test data with forecasts from every model for
visual comparison.  We filter to series **H1** for clarity.

In [ ]:
def tag(fc: pl.DataFrame, label: str) -> pl.DataFrame:
    """Add a 'model' column and rename y_hat → value."""
    return fc.with_columns(pl.lit(label).alias("model")).rename({"y_hat": "value"})


actuals = test.with_columns(pl.lit("Actual").alias("model")).rename({"y": "value"})

viz_df = pl.concat(
    [
        actuals,
        tag(fc_naive, "Naive"),
        tag(fc_snaive, "Seasonal Naive"),
        tag(fc_ma, "Moving Avg"),
        tag(fc_fft, "FFT"),
        tag(fc_ses, "SES"),
        tag(fc_holt, "Holt"),
        tag(fc_hw, "Holt-Winters"),
        tag(fc_arima, "ARIMA(1,1,1)"),
        tag(fc_auto, "Auto-ARIMA"),
        tag(fc_scum, "SCUM"),
    ],
    how="align",
).filter(pl.col("unique_id") == "H1")

viz_df.hvplot.line(
    x="ds",
    y="value",
    by="model",
    width=900,
    height=400,
    title="H1 — Actual vs Forecasts",
    ylabel="y",
)

## 8 — Evaluation Metrics

We compute five standard point-forecast metrics for every model:

| Metric | Scale-dependent? | Notes |
|---|---|---|
| MAE | Yes | Simple to interpret |
| RMSE | Yes | Penalises large errors |
| MAPE | No | Undefined when y = 0 |
| sMAPE | No | Symmetric variant |
| MASE | No | Relative to seasonal naive |

In [ ]:
forecasts = {
    "Naive": fc_naive,
    "Seasonal Naive": fc_snaive,
    "Moving Avg": fc_ma,
    "FFT": fc_fft,
    "SES": fc_ses,
    "Holt": fc_holt,
    "Holt-Winters": fc_hw,
    "ARIMA(1,1,1)": fc_arima,
    "Auto-ARIMA": fc_auto,
    "SCUM": fc_scum,
}

rows = []
for name, fc in forecasts.items():
    joined = test.join(fc, on=["unique_id", "ds"])
    m_mae = mae(joined, actual_col="y", predicted_col="y_hat", id_col="unique_id")
    m_rmse = rmse(joined, actual_col="y", predicted_col="y_hat", id_col="unique_id")
    m_mape = mape(joined, actual_col="y", predicted_col="y_hat", id_col="unique_id")
    m_smape = smape(joined, actual_col="y", predicted_col="y_hat", id_col="unique_id")
    m_mase = mase(joined, actual_col="y", predicted_col="y_hat", id_col="unique_id", season_length=SEASON_LENGTH)

    # Average across series for a single summary row
    rows.append(
        {
            "model": name,
            "MAE": m_mae["mae"].mean(),
            "RMSE": m_rmse["rmse"].mean(),
            "MAPE": m_mape["mape"].mean(),
            "sMAPE": m_smape["smape"].mean(),
            "MASE": m_mase["mase"].mean(),
        }
    )

metrics_df = pl.DataFrame(rows).sort("MAE")
metrics_df

In [ ]:
# Bar chart — MAE by model
metrics_df.hvplot.bar(
    x="model",
    y="MAE",
    rot=45,
    width=800,
    height=380,
    title="MAE by Model (averaged across H1, H2, H3)",
    color="steelblue",
)

## 9 — Cross-Validation with Expanding Window

Point estimates on a single split can be misleading.  `expanding_window_cv`
generates multiple (train, test) folds so we can assess forecast stability.

In [ ]:
cv_results = []

for fold_idx, (cv_train, cv_test) in enumerate(expanding_window_cv(df, n_splits=5, horizon=HORIZON)):
    fc = seasonal_naive_forecast(cv_train, h=HORIZON, season_length=SEASON_LENGTH)
    joined = cv_test.join(fc, on=["unique_id", "ds"])
    m = mae(joined, actual_col="y", predicted_col="y_hat", id_col="unique_id")
    cv_results.append(m.with_columns(pl.lit(fold_idx).alias("fold")))

cv_df = pl.concat(cv_results)
print("Seasonal Naive — MAE per fold (averaged across series):")
cv_df.group_by("fold").agg(pl.col("mae").mean()).sort("fold")

## 10 — Kaboudan Metric

The **Kaboudan metric** quantifies a time series' *predictability* by
measuring how much better a model performs relative to a random walk in
a backtesting loop.  Values near 1 indicate high predictability; values
near 0 suggest the series is essentially random.

We evaluate the SCUM model via the `.pts` namespace accessor.

In [ ]:
kaboudan_result = df.pts.kaboudan(
    sf,
    block_size=200,
    backtesting_start=0.5,
    n_folds=10,
)

kaboudan_result

## Summary

We progressed through **ten forecasting approaches** of increasing
sophistication:

1. **Baselines** (Naive, Seasonal Naive, Moving Average) set the floor.
2. **FFT** exploited dominant frequencies but assumed stationarity.
3. **Exponential Smoothing** (SES → Holt → Holt-Winters) adaptively
   weighted recent observations, adding trend and seasonality.
4. **ARIMA / Auto-ARIMA** modelled autocorrelation structure directly.
5. **SCUM** combined several statistical models into an ensemble.

Key takeaways:

- A seasonal baseline is often surprisingly competitive.
- Holt-Winters and Auto-ARIMA typically improve on baselines when
  seasonality and trend co-exist.
- Cross-validation gives a more honest picture than a single split.
- The Kaboudan metric tells us *whether* forecasting is worthwhile
  before we invest in complex models.

**Next notebook →** `04_machine_learning_forecasting.ipynb` will extend
this comparison to ML models (LightGBM, XGBoost) and global modelling.